# Sentiment & Emotion Analysis — Full Pipeline
Runs all 7 models, statistical tests, 3-class and binary evaluations, and paper figures.  
**Run cells top-to-bottom.** All outputs land in `data/results/` and `data/results/figures/`.

In [ ]:
%matplotlib inline
import os, re, gc, warnings, random
import pandas as pd
import numpy as np
import spacy
from spacytextblob.spacytextblob import SpacyTextBlob
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk import word_tokenize
import torch
from transformers import pipeline
import scikit_posthocs as sp_post
from scipy.stats.mstats import kruskal
from scipy.stats import friedmanchisquare, spearmanr
from scipy.stats import chi2 as _chi2
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, cohen_kappa_score, matthews_corrcoef,
    roc_auc_score, roc_curve, balanced_accuracy_score
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneGroupOut

warnings.filterwarnings('ignore')

# --- Reproducibility (revision round) ---
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
try:
    torch.use_deterministic_algorithms(True, warn_only=True)
except Exception:
    pass
os.environ['PYTHONHASHSEED'] = str(SEED)

nltk.download('vader_lexicon', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)


In [ ]:
BASE   = os.path.abspath(".")
DATA   = os.path.join(BASE, "data")
OUTDIR = os.path.join(DATA, "results")
FIGDIR = os.path.join(OUTDIR, "figures")
os.makedirs(OUTDIR, exist_ok=True)
os.makedirs(FIGDIR, exist_ok=True)

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(f"Device: {device}")

## 1. Load Data

In [ ]:
df = pd.read_csv(os.path.join(DATA, "forcaps_with_corrected.csv"))
df.drop(['SES','Uniqueness','Political Affiliation','Self Censorship',
         'Language Proficiency','Neurodivergence','Education','Race/Ethnicity'],
        axis=1, inplace=True)
df['corpus'] = df['Memory Title'].astype(str) + ': ' + df['Description'].astype(str)
print(f"Loaded {len(df)} rows")
df.head(3)

## 2. Sentiment Models
### 2.1 spaCy TextBlob

In [ ]:
nlp = spacy.load('en_core_web_sm')
nlp.add_pipe("spacytextblob")
doc = df['corpus'].apply(lambda x: nlp(x))
df['spaCy polarity'] = [x._.blob.polarity for x in doc]
del nlp, doc; gc.collect()
print("spaCy done.")

### 2.2 NLTK VADER

In [ ]:
analyzer = SentimentIntensityAnalyzer()
df['nltk polarity'] = df['corpus'].apply(
    lambda t: analyzer.polarity_scores(t)['compound'])
print("NLTK VADER done.")

### 2.3 HuggingFace Models
Helper functions for normalizing raw model outputs to a 0–1 polarity scale.

In [ ]:
def normalize_stars(dict_list):
    temp = pd.DataFrame(dict_list)
    if len(dict_list) == 5:
        order = ['5 stars','4 stars','3 stars','2 stars','1 star']
        temp['label'] = pd.Categorical(temp['label'], categories=order, ordered=True)
        ordered = temp.sort_values('label')
        total, scaler, adj = [], 1, 0
        for s in ordered['score']:
            total.append(s * (scaler - adj)); adj += 0.25
        return sum(total)
    if len(dict_list) == 3:
        pos = temp[temp['label'].isin(['positive','Positive'])]['score'].values
        neg = temp[temp['label'].isin(['negative','Negative'])]['score'].values
        neu = temp[temp['label'].isin(['neutral','Neutral'])]['score'].values
        return float(sum([pos*1.0, neu*0.5, neg*0]))
    if len(dict_list) == 2:
        pos = temp[temp['label'].isin(['positive','Positive'])]['score'].values
        neg = temp[temp['label'].isin(['negative','Negative'])]['score'].values
        return float(sum([pos*1, neg*0]))

def normalize_tabmodel(dict_list):
    total, temp = [], pd.DataFrame(dict_list)
    order = ['Very Positive','Positive','Neutral','Negative','Very Negative']
    temp['label'] = pd.Categorical(temp['label'], categories=order, ordered=True)
    ordered = temp.sort_values('label')
    scaler, adj = 1, 0
    for s in ordered['score']:
        total.append(s * (scaler - adj)); adj += 0.25
    return sum(total)

desc_list = df['corpus'].astype(str).tolist()
print(f"Prepared {len(desc_list)} texts for inference.")

#### nlptown/bert-base-multilingual-uncased-sentiment (5-star)

In [ ]:
m0 = pipeline("text-classification",
               model="nlptown/bert-base-multilingual-uncased-sentiment",
               top_k=5, batch_size=64, device=device)
out0 = m0(desc_list)
assert len(out0) == len(df)
del m0; gc.collect(); torch.cuda.empty_cache()

cat0 = lambda x: 'positive' if x in ['5 stars','4 stars'] else 'neutral' if x == '3 stars' else 'negative'
df['nlptown_polarity']  = [normalize_stars(d) for d in out0]
df['nlptown_sentiment'] = [cat0(d[0]['label']) for d in out0]
del out0
print("nlptown done.")

#### cardiffnlp/twitter-roberta-base-sentiment-latest

In [ ]:
m1 = pipeline("text-classification",
               model="cardiffnlp/twitter-roberta-base-sentiment-latest",
               top_k=3, batch_size=64, device=device)
out1 = m1(desc_list)
assert len(out1) == len(df)
del m1; gc.collect(); torch.cuda.empty_cache()

df['cardiff_polarity']  = [normalize_stars(d) for d in out1]
df['cardiff_sentiment'] = [d[0]['label'] for d in out1]
del out1
print("Cardiff done.")

#### tabularisai/multilingual-sentiment-analysis

In [ ]:
m2 = pipeline("text-classification",
               model="tabularisai/multilingual-sentiment-analysis",
               top_k=5, batch_size=96, device=device)
out2 = m2(desc_list)
assert len(out2) == len(df)
del m2; gc.collect(); torch.cuda.empty_cache()

cat2 = lambda x: 'positive' if x in ['Very Positive','Positive'] else 'neutral' if x == 'Neutral' else 'negative'
df['tabmulti_polarity']  = [round(normalize_tabmodel(d), 4) for d in out2]
df['tabmulti_sentiment'] = [cat2(d[0]['label']) for d in out2]
del out2
print("tabmulti done.")

#### j-hartmann/emotion-english-distilroberta-base

In [ ]:
# Valence weights derived from Mohammad (2018) NRC Valence-Arousal-Dominance
# (NRC-VAD) lexicon. Each emotion word's valence dimension is taken directly
# from NRC-VAD (range [0, 1], higher = more positive). 'neutral' is set to
# 0.5 as a midpoint since it is not a basic emotion in NRC-VAD.
# Reference: Mohammad, S. (2018). Obtaining Reliable Human Ratings of
#            Valence, Arousal, and Dominance for 20,000 English Words. ACL.
HARTMANN_VALENCE = {
    'joy':      0.980,   # NRC-VAD
    'surprise': 0.875,   # NRC-VAD
    'neutral':  0.500,   # midpoint (not in NRC-VAD as basic emotion)
    'fear':     0.073,   # NRC-VAD
    'sadness':  0.052,   # NRC-VAD
    'disgust':  0.052,   # NRC-VAD
    'anger':    0.167,   # NRC-VAD
}
HARTMANN_POS = {'joy', 'surprise'}

m3 = pipeline("text-classification",
               model="j-hartmann/emotion-english-distilroberta-base",
               top_k=7, batch_size=64, device=device)
out3 = m3(desc_list)
assert len(out3) == len(df)
del m3; gc.collect(); torch.cuda.empty_cache()

def norm_hartmann(dl):
    return round(sum(d['score'] * HARTMANN_VALENCE[d['label']] for d in dl), 4)

df['hartmann_polarity']  = [norm_hartmann(d) for d in out3]
df['hartmann_sentiment'] = [
    'positive' if d[0]['label'] in HARTMANN_POS else
    'neutral'  if d[0]['label'] == 'neutral' else
    'negative' for d in out3]
# Keep raw outputs for sensitivity analysis later
HARTMANN_RAW = out3
print("Hartmann done.")


#### bhadresh-savani/distilbert-base-uncased-emotion

In [ ]:
# Bhadresh valence weights also taken from NRC-VAD (Mohammad, 2018).
BHADRESH_VALENCE = {
    'joy':      0.980,
    'love':     1.000,
    'surprise': 0.875,
    'fear':     0.073,
    'sadness':  0.052,
    'anger':    0.167,
}
BHADRESH_POS = {'joy', 'love', 'surprise'}

m4 = pipeline("text-classification",
               model="bhadresh-savani/distilbert-base-uncased-emotion",
               top_k=6, batch_size=64, device=device)
out4 = m4(desc_list)
assert len(out4) == len(df)
del m4; gc.collect(); torch.cuda.empty_cache()

def norm_bhadresh(dl):
    return round(sum(d['score'] * BHADRESH_VALENCE[d['label']] for d in dl), 4)

df['bhadresh_polarity']  = [norm_bhadresh(d) for d in out4]
BHADRESH_RAW = out4

# --- Sweep neutral-confidence threshold and pick the best by 3-class macro F1
# Need ground-truth labels for this; use a temporary discretization.
disc_conv_tmp = lambda x: 'positive' if x >= 0.6667 else 'negative' if x <= 0.3333 else 'neutral'
y_true_tmp = df['Emotion'].apply(lambda x: (x - 1) / 6).apply(disc_conv_tmp).values

best_thr, best_f1 = 0.4, -1
sweep_records = []
for thr in np.arange(0.30, 0.61, 0.05):
    preds = []
    for d in BHADRESH_RAW:
        if d[0]['score'] < thr:
            preds.append('neutral')
        elif d[0]['label'] in BHADRESH_POS:
            preds.append('positive')
        else:
            preds.append('negative')
    f1 = f1_score(y_true_tmp, preds, average='macro', zero_division=0)
    sweep_records.append({'threshold': round(thr, 2), 'macro_f1': round(f1, 4)})
    if f1 > best_f1:
        best_f1, best_thr = f1, thr

NEUTRAL_THRESHOLD = round(best_thr, 2)
sweep_df = pd.DataFrame(sweep_records)
print('Bhadresh neutral-threshold sweep:')
print(sweep_df.to_string(index=False))
print(f'Selected NEUTRAL_THRESHOLD = {NEUTRAL_THRESHOLD} (best macro F1 = {best_f1:.4f})')
sweep_df.to_csv(os.path.join(OUTDIR, 'bhadresh_threshold_sweep.csv'), index=False)

df['bhadresh_sentiment'] = [
    'neutral'  if d[0]['score'] < NEUTRAL_THRESHOLD else
    'positive' if d[0]['label'] in BHADRESH_POS else
    'negative' for d in BHADRESH_RAW]
print("Bhadresh done.")


## 3. Normalize & Discretize
Scale all continuous scores to [0, 1].  
Discretize to positive / neutral / negative using thresholds ≥0.667 and ≤0.333.

In [ ]:
min_max_sent = lambda x: (x - (-1)) / 2
min_max_emot = lambda x: (x - 1) / 6

df['spaCy_polarity'] = df['spaCy polarity'].apply(min_max_sent)
df['nltk_polarity']  = df['nltk polarity'].apply(min_max_sent)
df.drop(['spaCy polarity','nltk polarity'], axis=1, inplace=True)
df['participant_sentiment'] = df['Emotion'].apply(min_max_emot)

disc_conv = lambda x: 'positive' if x >= 0.6667 else 'negative' if x <= 0.3333 else 'neutral'
df['spaCy_sentiment'] = df['spaCy_polarity'].apply(disc_conv)
df['nltk_sentiment']  = df['nltk_polarity'].apply(disc_conv)

df.drop(['Centrality','Vividness','Rehearsal','Age in Memory','Ordinal AiM'],
        axis=1, inplace=True)

df.to_csv(os.path.join(OUTDIR, "memories_with_scores.csv"), index=False)
print("Saved memories_with_scores.csv")
df.head(3)

## 4. Statistical Comparison (Continuous Polarity)
Kruskal-Wallis (reference) + Dunnett's test comparing each model's polarity to participant ratings.

In [ ]:
score_cols = ['participant_sentiment','spaCy_polarity','nltk_polarity',
              'nlptown_polarity','cardiff_polarity','tabmulti_polarity',
              'hartmann_polarity','bhadresh_polarity']

# Friedman omnibus on continuous polarity scores (paired across memories)
res_f_cont = friedmanchisquare(*[df[c].values for c in score_cols])
print(f'Friedman (continuous polarity): chi2={res_f_cont.statistic:.4f}, p={res_f_cont.pvalue:.4e}')

# Conover post-hoc (paired) — proper post-hoc after Friedman.
# Replaces the previously-used Dunnett's test, which assumes independence
# and therefore was inappropriate for this within-memory paired design.
cont_long = pd.melt(df[score_cols].reset_index(),
                    id_vars='index', value_vars=score_cols,
                    var_name='measure', value_name='value')
posthoc_cont = sp_post.posthoc_conover_friedman(
    df[score_cols].values, p_adjust='holm')
posthoc_cont.columns = score_cols
posthoc_cont.index   = score_cols
posthoc_cont.to_csv(os.path.join(OUTDIR, 'conover_continuous.csv'))

# Pull comparisons of each model vs participant
model_score_cols = score_cols[1:]
conover_cont_df = pd.DataFrame({
    'Comparison':     [f"Participant vs {c.split('_')[0]}" for c in model_score_cols],
    'Mean Model':     [round(df[c].mean(), 4) for c in model_score_cols],
    'Mean Participant': [round(df['participant_sentiment'].mean(), 4)] * len(model_score_cols),
    'Mean Diff':      [round(df['participant_sentiment'].mean() - df[c].mean(), 4) for c in model_score_cols],
    'Conover p (Holm-adj)': [posthoc_cont.iloc[0, i+1] for i in range(len(model_score_cols))],
})
conover_cont_df.to_csv(os.path.join(OUTDIR, 'conover_continuous_vs_participant.csv'), index=False)
print('Conover continuous (vs participant, Holm-adjusted):')
print(conover_cont_df.to_string(index=False))


## 5. Prepare Categorical Labels & Linguistic Features

In [ ]:
# ---- Variable naming convention (clarified in revision):
#   df['participant_score']     = continuous normalised [0,1]
#   df['participant_polarity']  = alias of participant_score (kept for BC)
#   df['participant_sentiment'] = discretised 3-class label (negative/neutral/positive)
df['participant_score']     = df['participant_sentiment'].copy()        # continuous
df['participant_polarity']  = df['participant_score']                   # alias
df['participant_sentiment'] = df['participant_score'].apply(disc_conv)  # discrete

model_sentiment_cols = ['spaCy_sentiment','nltk_sentiment','nlptown_sentiment',
                        'cardiff_sentiment','tabmulti_sentiment',
                        'hartmann_sentiment','bhadresh_sentiment']

discretized_df = df[['corpus','participant_sentiment'] + model_sentiment_cols]

def count_pos(text, tag):
    return sum(1 for _,p in nltk.pos_tag(word_tokenize(text)) if p.startswith(tag))

df['adj_count']  = df['corpus'].apply(lambda x: count_pos(x,'JJ'))
df['noun_count'] = df['corpus'].apply(lambda x: count_pos(x,'NN'))
df['verb_count'] = df['corpus'].apply(lambda x: count_pos(x,'VB'))
df['word_count'] = df['corpus'].apply(lambda x: len(re.findall(r'\b\w+\b', x.lower())))
df['char_count'] = df['corpus'].apply(len)
print("Linguistic features computed.")


## 6. 3-Class Evaluation (negative / neutral / positive)

In [ ]:
ranker = lambda x: 1 if x=='negative' else 2 if x=='neutral' else 3
cols_ord = ['participant_sentiment'] + model_sentiment_cols
test_df = discretized_df[['corpus'] + cols_ord].copy()
for c in cols_ord:
    test_df[c] = test_df[c].apply(ranker)

label_counts = Counter(test_df['participant_sentiment'])
total = test_df.shape[0]
rank_name = {1:'negative', 2:'neutral', 3:'positive'}
model_names = ['spaCy','nltk','nlptown','cardiff','tabmulti','hartmann','bhadresh']

print("Participant sentiment distribution:")
for r in sorted(label_counts):
    print(f"  {rank_name[r]:8s}: {label_counts[r]:4d}  ({label_counts[r]/total:.1%})")

always_pos_acc = label_counts[3] / total
majority_rank  = label_counts.most_common(1)[0][0]
majority_acc   = label_counts[majority_rank] / total
print(f"\nAlways-positive baseline : {always_pos_acc:.4f} ({always_pos_acc:.1%})")
print(f"Majority-class baseline  : {majority_acc:.4f} ({majority_acc:.1%})  [always '{rank_name[majority_rank]}']")

acc_list = [test_df[test_df['participant_sentiment']==test_df[c]].shape[0]/total
            for c in cols_ord]

print(f"\n{'Model':12s}  {'Accuracy':>8s}  {'vs always-positive':>20s}")
print('-' * 45)
for name, acc in zip(model_names, acc_list[1:]):
    print(f"{name:12s}  {acc:8.4f}  {acc - always_pos_acc:+.1%}")

# --- Friedman (omnibus) + Conover post-hoc, paired across memories.
# Replaces the previously-used Dunnett, which assumes independence.
res_f = friedmanchisquare(*[test_df[c].values for c in cols_ord])
print(f"\nFriedman chi-square (3-class ranks): {res_f.statistic:.4f}, p={res_f.pvalue:.4e}")

posthoc_disc = sp_post.posthoc_conover_friedman(
    test_df[cols_ord].values, p_adjust='holm')
posthoc_disc.columns = cols_ord
posthoc_disc.index   = cols_ord
posthoc_disc.to_csv(os.path.join(OUTDIR, 'conover_discretized.csv'))

conover_disc_df = pd.DataFrame({
    'Comparison':     [f"Participant vs {c.split('_')[0]}" for c in cols_ord[1:]],
    'Accuracy':       acc_list[1:],
    'Mean Diff':      [round(test_df['participant_sentiment'].mean() - test_df[c].mean(), 4)
                       for c in cols_ord[1:]],
    'Conover p (Holm-adj)': [posthoc_disc.iloc[0, i+1] for i in range(len(cols_ord)-1)],
})
conover_disc_df.to_csv(os.path.join(OUTDIR, 'conover_discretized_vs_participant.csv'), index=False)
print('\nConover (3-class ranks vs participant, Holm-adjusted):')
print(conover_disc_df.to_string(index=False))


## 7. Binary Evaluation (positive vs not-positive)
Neutral and negative are merged into **not_positive**.

In [ ]:
bin_conv = lambda x: 'positive' if x == 'positive' else 'not_positive'

bin_cols_ord = ['participant_sentiment'] + model_sentiment_cols
bin_test_df = discretized_df[['corpus'] + bin_cols_ord].copy()
for c in bin_cols_ord:
    bin_test_df[c] = bin_test_df[c].apply(bin_conv)

bin_ranker = lambda x: 1 if x == 'positive' else 0
for c in bin_cols_ord:
    bin_test_df[c] = bin_test_df[c].apply(bin_ranker)

bin_label_counts = Counter(bin_test_df['participant_sentiment'])
bin_rank_name = {1: 'positive', 0: 'not_positive'}
print("Participant binary sentiment distribution:")
for r in sorted(bin_label_counts, reverse=True):
    print(f"  {bin_rank_name[r]:12s}: {bin_label_counts[r]:4d}  ({bin_label_counts[r]/total:.1%})")

always_pos_bin   = bin_label_counts[1] / total
maj_bin_rank     = bin_label_counts.most_common(1)[0][0]
majority_bin_acc = bin_label_counts[maj_bin_rank] / total
print(f"\nAlways-positive baseline : {always_pos_bin:.4f} ({always_pos_bin:.1%})")
print(f"Majority-class baseline  : {majority_bin_acc:.4f} ({majority_bin_acc:.1%})  [always '{bin_rank_name[maj_bin_rank]}']")

bin_acc_list = [
    bin_test_df[bin_test_df['participant_sentiment'] == bin_test_df[c]].shape[0] / total
    for c in bin_cols_ord
]

print(f"\n{'Model':12s}  {'Binary Acc':>10s}  {'3-class Acc':>11s}  {'vs majority':>12s}")
print('-' * 52)
for name, bacc, acc3 in zip(model_names, bin_acc_list[1:], acc_list[1:]):
    print(f"{name:12s}  {bacc:10.4f}  {acc3:11.4f}  {bacc - majority_bin_acc:+.1%}")

res_fb = friedmanchisquare(*[bin_test_df[c].values for c in bin_cols_ord])
print(f"\nFriedman chi-square (binary): {res_fb.statistic:.4f}, p={res_fb.pvalue:.4e}")

posthoc_bin = sp_post.posthoc_conover_friedman(
    bin_test_df[bin_cols_ord].values, p_adjust='holm')
posthoc_bin.columns = bin_cols_ord
posthoc_bin.index   = bin_cols_ord
posthoc_bin.to_csv(os.path.join(OUTDIR, 'conover_binary.csv'))

conover_bin_df = pd.DataFrame({
    'Comparison':      [f"Participant vs {c.split('_')[0]}" for c in model_sentiment_cols],
    'Binary Accuracy': bin_acc_list[1:],
    '3class Accuracy': acc_list[1:],
    'Mean Diff':       [round(bin_test_df['participant_sentiment'].mean() - bin_test_df[c].mean(), 4)
                        for c in model_sentiment_cols],
    'Conover p (Holm-adj)': [posthoc_bin.iloc[0, i+1] for i in range(len(model_sentiment_cols))],
})
conover_bin_df.to_csv(os.path.join(OUTDIR, 'conover_binary_vs_participant.csv'), index=False)
print('\nConover binary (vs participant, Holm-adjusted):')
print(conover_bin_df.to_string(index=False))


## 8. Error Analysis Files

In [ ]:
# 3-class
sub_df = df[['corpus','char_count','word_count','verb_count','noun_count','adj_count',
             'participant_sentiment'] + model_sentiment_cols].copy()
for m in model_sentiment_cols:
    sub_df[f"clf_{m}"] = sub_df.apply(
        lambda r: 'correct' if r['participant_sentiment']==r[m] else 'misclassified', axis=1)
sub_df.to_csv(os.path.join(OUTDIR, "for_error_analysis.csv"), index=False)

long_df = sub_df.melt(
    id_vars=["corpus","char_count","word_count","verb_count","noun_count","adj_count"],
    value_vars=[f"clf_{m}" for m in model_sentiment_cols],
    var_name="model", value_name="sentiment")
long_df['sentiment'] = long_df['sentiment'].apply(lambda x: 1 if x=='correct' else 0)
long_df.to_csv(os.path.join(OUTDIR, "long_for_error_analysis.csv"), index=False)

# binary
bin_sub_df = df[['corpus','char_count','word_count','verb_count','noun_count','adj_count',
                 'participant_sentiment'] + model_sentiment_cols].copy()
for c in ['participant_sentiment'] + model_sentiment_cols:
    bin_sub_df[c] = bin_sub_df[c].apply(bin_conv)
for m in model_sentiment_cols:
    bin_sub_df[f"clf_{m}"] = bin_sub_df.apply(
        lambda r, m=m: 'correct' if r['participant_sentiment'] == r[m] else 'misclassified', axis=1)
bin_sub_df.to_csv(os.path.join(OUTDIR, "for_error_analysis_binary.csv"), index=False)

bin_long_df = bin_sub_df.melt(
    id_vars=["corpus","char_count","word_count","verb_count","noun_count","adj_count"],
    value_vars=[f"clf_{m}" for m in model_sentiment_cols],
    var_name="model", value_name="sentiment")
bin_long_df['sentiment'] = bin_long_df['sentiment'].apply(lambda x: 1 if x == 'correct' else 0)
bin_long_df.to_csv(os.path.join(OUTDIR, "long_for_error_analysis_binary.csv"), index=False)

print("Saved: for_error_analysis.csv, long_for_error_analysis.csv")
print("Saved: for_error_analysis_binary.csv, long_for_error_analysis_binary.csv")

## 9. Paper Metrics & Figures
Computes full metric tables and saves 6 publication-quality figures (PDF + PNG) to `results/figures/`.

In [ ]:
plt.rcParams.update({'font.size': 9, 'axes.titlesize': 10, 'axes.labelsize': 9})

labels_3       = ['negative', 'neutral', 'positive']
labels_bin_str = ['not_positive', 'positive']
display_names  = ['spaCy', 'NLTK', 'nlptown', 'Cardiff', 'tabmulti', 'Hartmann', 'Bhadresh']
polarity_cols  = ['spaCy_polarity', 'nltk_polarity', 'nlptown_polarity',
                  'cardiff_polarity', 'tabmulti_polarity', 'hartmann_polarity', 'bhadresh_polarity']

y_true_3   = discretized_df['participant_sentiment'].values
y_true_bin = np.array([bin_conv(y) for y in y_true_3])
y_true_b01 = (y_true_bin == 'positive').astype(int)
participant_cont = df['participant_polarity'].values  # continuous [0,1]

rows_3, rows_bin = [], []
for name, col, pol_col in zip(display_names, model_sentiment_cols, polarity_cols):
    y_pred_3   = discretized_df[col].values
    y_pred_bin = np.array([bin_conv(y) for y in y_pred_3])
    y_pred_b01 = (y_pred_bin == 'positive').astype(int)
    pol        = df[pol_col].values

    f1_per   = f1_score(y_true_3, y_pred_3, labels=labels_3, average=None, zero_division=0)
    prec_per = precision_score(y_true_3, y_pred_3, labels=labels_3, average=None, zero_division=0)
    rec_per  = recall_score(y_true_3, y_pred_3, labels=labels_3, average=None, zero_division=0)

    sp_r, sp_p = spearmanr(participant_cont, pol)

    rows_3.append({
        'Model':          name,
        'Accuracy':       round(accuracy_score(y_true_3, y_pred_3), 4),
        'Balanced Acc':   round(balanced_accuracy_score(y_true_3, y_pred_3), 4),
        'Macro P':        round(precision_score(y_true_3, y_pred_3, average='macro', zero_division=0), 4),
        'Macro R':        round(recall_score(y_true_3, y_pred_3, average='macro', zero_division=0), 4),
        'Macro F1':       round(f1_score(y_true_3, y_pred_3, average='macro', zero_division=0), 4),
        'Weighted F1':    round(f1_score(y_true_3, y_pred_3, average='weighted', zero_division=0), 4),
        'Kappa':          round(cohen_kappa_score(y_true_3, y_pred_3), 4),
        'Weighted Kappa': round(cohen_kappa_score(y_true_3, y_pred_3, weights='linear'), 4),
        'MCC':            round(matthews_corrcoef(y_true_3, y_pred_3), 4),
        'Spearman r':     round(sp_r, 4),
        'Spearman p':     sp_p,                          # raw p (use sci notation in tables)
        'P negative':     round(prec_per[0], 4),
        'P neutral':      round(prec_per[1], 4),
        'P positive':     round(prec_per[2], 4),
        'R negative':     round(rec_per[0], 4),
        'R neutral':      round(rec_per[1], 4),
        'R positive':     round(rec_per[2], 4),
        'F1 negative':    round(f1_per[0], 4),
        'F1 neutral':     round(f1_per[1], 4),
        'F1 positive':    round(f1_per[2], 4),
    })

    prec_bin = precision_score(y_true_bin, y_pred_bin, labels=labels_bin_str, average=None, zero_division=0)
    rec_bin  = recall_score(y_true_bin, y_pred_bin, labels=labels_bin_str, average=None, zero_division=0)

    cm_b = confusion_matrix(y_true_b01, y_pred_b01)
    tn, fp, fn, tp = cm_b.ravel()
    specificity = round(tn / (tn + fp), 4) if (tn + fp) > 0 else 0.0

    # Note: in 2-class case, weighted kappa with linear weights == unweighted kappa,
    # so we omit Weighted Kappa from the binary table to avoid duplication.
    rows_bin.append({
        'Model':          name,
        'Accuracy':       round(accuracy_score(y_true_bin, y_pred_bin), 4),
        'Balanced Acc':   round(balanced_accuracy_score(y_true_bin, y_pred_bin), 4),
        'Precision':      round(precision_score(y_true_b01, y_pred_b01, zero_division=0), 4),
        'Recall':         round(recall_score(y_true_b01, y_pred_b01, zero_division=0), 4),
        'Specificity':    specificity,
        'F1 positive':    round(f1_score(y_true_b01, y_pred_b01, zero_division=0), 4),
        'Macro F1':       round(f1_score(y_true_bin, y_pred_bin, average='macro', zero_division=0), 4),
        'Weighted F1':    round(f1_score(y_true_bin, y_pred_bin, average='weighted', zero_division=0), 4),
        'Kappa':          round(cohen_kappa_score(y_true_bin, y_pred_bin), 4),
        'MCC':            round(matthews_corrcoef(y_true_bin, y_pred_bin), 4),
        'Spearman r':     round(sp_r, 4),
        'Spearman p':     sp_p,
        'AUC-ROC':        round(roc_auc_score(y_true_b01, pol), 4),
        'P not_positive': round(prec_bin[0], 4),
        'P positive':     round(prec_bin[1], 4),
        'R not_positive': round(rec_bin[0], 4),
        'R positive':     round(rec_bin[1], 4),
    })

metrics_3_df   = pd.DataFrame(rows_3)
metrics_bin_df = pd.DataFrame(rows_bin)
metrics_3_df.to_csv(os.path.join(OUTDIR, 'metrics_3class.csv'), index=False)
metrics_bin_df.to_csv(os.path.join(OUTDIR, 'metrics_binary.csv'), index=False)
print('Saved metrics_3class.csv and metrics_binary.csv')
print('\n3-class:')
display(metrics_3_df)
print('\nBinary:')
display(metrics_bin_df)


### Figure 1: 3-Class Confusion Matrices

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes = axes.flatten()
for i, (name, col) in enumerate(zip(display_names, model_sentiment_cols)):
    cm = confusion_matrix(y_true_3, discretized_df[col].values, labels=labels_3, normalize='true')
    sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues', ax=axes[i],
                xticklabels=labels_3, yticklabels=labels_3, vmin=0, vmax=1,
                cbar=(i == 6))
    axes[i].set_title(name)
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('True' if i % 4 == 0 else '')
axes[-1].set_visible(False)
fig.suptitle('3-Class Confusion Matrices (Row-Normalized)', fontsize=12, fontweight='bold')
plt.tight_layout()
fig.savefig(os.path.join(FIGDIR, 'confusion_matrices_3class.pdf'), bbox_inches='tight')
fig.savefig(os.path.join(FIGDIR, 'confusion_matrices_3class.png'), dpi=300, bbox_inches='tight')
plt.show()

### Figure 2: Binary Confusion Matrices

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
axes = axes.flatten()
for i, (name, col) in enumerate(zip(display_names, model_sentiment_cols)):
    y_pred_bin_i = np.array([bin_conv(y) for y in discretized_df[col].values])
    cm = confusion_matrix(y_true_bin, y_pred_bin_i, labels=labels_bin_str, normalize='true')
    sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues', ax=axes[i],
                xticklabels=labels_bin_str, yticklabels=labels_bin_str, vmin=0, vmax=1,
                cbar=(i == 6))
    axes[i].set_title(name)
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('True' if i % 4 == 0 else '')
axes[-1].set_visible(False)
fig.suptitle('Binary Confusion Matrices (Row-Normalized)', fontsize=12, fontweight='bold')
plt.tight_layout()
fig.savefig(os.path.join(FIGDIR, 'confusion_matrices_binary.pdf'), bbox_inches='tight')
fig.savefig(os.path.join(FIGDIR, 'confusion_matrices_binary.png'), dpi=300, bbox_inches='tight')
plt.show()

### Figure 3: Metrics Comparison (3-class vs Binary)

In [ ]:
metrics_shared = ['Accuracy', 'Macro F1', 'Weighted F1', 'Kappa', 'MCC']
x      = np.arange(len(display_names))
width  = 0.14
colors = plt.cm.tab10(np.linspace(0, 0.55, len(metrics_shared)))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, mdf, title, ap_base, maj_base in zip(
        axes,
        [metrics_3_df, metrics_bin_df],
        ['3-Class', 'Binary (positive vs not-positive)'],
        [always_pos_acc, always_pos_bin],
        [majority_acc, majority_bin_acc]):
    for j, metric in enumerate(metrics_shared):
        ax.bar(x + j * width, mdf[metric], width, label=metric, color=colors[j])
    ax.axhline(ap_base,  color='gray', linestyle='--', linewidth=0.9, label='always-positive')
    ax.axhline(maj_base, color='red',  linestyle='--', linewidth=0.9, label='majority baseline')
    ax.set_xticks(x + width * (len(metrics_shared) - 1) / 2)
    ax.set_xticklabels(display_names, rotation=30, ha='right')
    ax.set_ylim(0, 1.1)
    ax.set_ylabel('Score')
    ax.set_title(title, fontweight='bold')
    ax.legend(fontsize=7, loc='upper right', ncol=2)
fig.suptitle('Model Performance Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(os.path.join(FIGDIR, 'metrics_comparison.pdf'), bbox_inches='tight')
fig.savefig(os.path.join(FIGDIR, 'metrics_comparison.png'), dpi=300, bbox_inches='tight')
plt.show()

### Figure 4: ROC Curves (Binary)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for name, pol_col in zip(display_names, polarity_cols):
    pol = df[pol_col].values
    fpr, tpr, _ = roc_curve(y_true_b01, pol)
    auc = roc_auc_score(y_true_b01, pol)
    ax.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")
ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Binary (positive vs not-positive)', fontweight='bold')
ax.legend(fontsize=8, loc='lower right')
ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
plt.tight_layout()
fig.savefig(os.path.join(FIGDIR, 'roc_curves_binary.pdf'), bbox_inches='tight')
fig.savefig(os.path.join(FIGDIR, 'roc_curves_binary.png'), dpi=300, bbox_inches='tight')
plt.show()

### Figure 5: Per-Class F1 Heatmap (3-Class)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

per_class_data = {
    'Precision': metrics_3_df[['Model','P negative','P neutral','P positive']].set_index('Model').rename(columns={'P negative':'Negative','P neutral':'Neutral','P positive':'Positive'}),
    'Recall':    metrics_3_df[['Model','R negative','R neutral','R positive']].set_index('Model').rename(columns={'R negative':'Negative','R neutral':'Neutral','R positive':'Positive'}),
    'F1':        metrics_3_df[['Model','F1 negative','F1 neutral','F1 positive']].set_index('Model').rename(columns={'F1 negative':'Negative','F1 neutral':'Neutral','F1 positive':'Positive'}),
}

for ax, (metric, data) in zip(axes, per_class_data.items()):
    sns.heatmap(data, annot=True, fmt='.3f', cmap='RdYlGn', ax=ax, vmin=0, vmax=1,
                cbar=(metric == 'F1'))
    ax.set_title(f'Per-Class {metric} — 3-Class', fontweight='bold')
    ax.set_xlabel('Sentiment Class')
    ax.set_ylabel('Model' if metric == 'Precision' else '')

plt.tight_layout()
fig.savefig(os.path.join(FIGDIR, 'per_class_prf_heatmap.pdf'), bbox_inches='tight')
fig.savefig(os.path.join(FIGDIR, 'per_class_prf_heatmap.png'), dpi=300, bbox_inches='tight')
plt.show()

### Figure 6: Accuracy & Macro F1 Gain (3-class → Binary)

In [ ]:
acc_gain = [b - a for b, a in zip(metrics_bin_df['Accuracy'], metrics_3_df['Accuracy'])]
f1_gain  = [b - a for b, a in zip(metrics_bin_df['Macro F1'], metrics_3_df['Macro F1'])]
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - 0.2, acc_gain, 0.35, label='Accuracy gain', color='steelblue')
ax.bar(x + 0.2, f1_gain,  0.35, label='Macro F1 gain', color='tomato')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(x); ax.set_xticklabels(display_names, rotation=30, ha='right')
ax.set_ylabel('Score gain (binary − 3-class)')
ax.set_title('Gain from Collapsing to Binary Classification', fontweight='bold')
ax.legend()
plt.tight_layout()
fig.savefig(os.path.join(FIGDIR, 'gain_binary_vs_3class.pdf'), bbox_inches='tight')
fig.savefig(os.path.join(FIGDIR, 'gain_binary_vs_3class.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"\nAll outputs saved to {OUTDIR}")

### Figure 7: Spearman Correlation (continuous polarity vs participant)

In [ ]:
sp_r_vals = metrics_3_df['Spearman r'].values
sp_p_vals = metrics_3_df['Spearman p'].values
colors_sp = ['steelblue' if p < 0.05 else 'lightsteelblue' for p in sp_p_vals]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(display_names, sp_r_vals, color=colors_sp, edgecolor='black', linewidth=0.6)

for bar, r, p in zip(bars, sp_r_vals, sp_p_vals):
    label = f'{r:.3f}' + ('*' if p < 0.05 else '')
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            label, ha='center', va='bottom', fontsize=8)

ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel("Spearman r (model polarity vs participant rating)")
ax.set_title('Spearman Correlation — Continuous Polarity vs Participant Emotion', fontweight='bold')
ax.set_ylim(min(sp_r_vals) - 0.1, max(sp_r_vals) + 0.1)
ax.text(0.98, 0.02, '* p < 0.05  (solid = significant)', transform=ax.transAxes,
        ha='right', va='bottom', fontsize=7, color='gray')
plt.tight_layout()
fig.savefig(os.path.join(FIGDIR, 'spearman_correlation.pdf'), bbox_inches='tight')
fig.savefig(os.path.join(FIGDIR, 'spearman_correlation.png'), dpi=300, bbox_inches='tight')
plt.show()


### Figure 8: Prediction Distribution vs Ground Truth

In [ ]:
labels_display = ['Negative', 'Neutral', 'Positive']
dist_rows = []
dist_rows.append({'Model': 'Participant',
                  'Negative': sum(y_true_3=='negative')/len(y_true_3),
                  'Neutral':  sum(y_true_3=='neutral')/len(y_true_3),
                  'Positive': sum(y_true_3=='positive')/len(y_true_3)})
for name, col in zip(display_names, model_sentiment_cols):
    preds = discretized_df[col].values
    dist_rows.append({'Model': name,
                      'Negative': sum(preds=='negative')/len(preds),
                      'Neutral':  sum(preds=='neutral')/len(preds),
                      'Positive': sum(preds=='positive')/len(preds)})
dist_df = pd.DataFrame(dist_rows).set_index('Model')

fig, ax = plt.subplots(figsize=(11, 5))
x_pos = np.arange(len(dist_df))
width = 0.25
colors_d = {'Negative': '#d62728', 'Neutral': '#aec7e8', 'Positive': '#2ca02c'}
for j, label in enumerate(labels_display):
    ax.bar(x_pos + (j-1)*width, dist_df[label], width,
           label=label, color=colors_d[label], alpha=0.85, edgecolor='black', linewidth=0.4)
ax.set_xticks(x_pos)
ax.set_xticklabels(dist_df.index, rotation=30, ha='right')
ax.set_ylabel('Proportion of predictions')
ax.set_title('Prediction Distribution: Models vs Participant Ground Truth', fontweight='bold')
ax.legend(title='Sentiment')
ax.axvline(0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
plt.tight_layout()
fig.savefig(os.path.join(FIGDIR, 'prediction_distribution.pdf'), bbox_inches='tight')
fig.savefig(os.path.join(FIGDIR, 'prediction_distribution.png'), dpi=300, bbox_inches='tight')
plt.show()

### Figure 9: Polarity Score Distributions by True Participant Class

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()
pal = {'negative': '#d62728', 'neutral': '#aec7e8', 'positive': '#2ca02c'}
for i, (name, pol_col) in enumerate(zip(display_names, polarity_cols)):
    vplot_df = pd.DataFrame({'Polarity': df[pol_col].values, 'True Label': y_true_3})
    vplot_df['True Label'] = pd.Categorical(vplot_df['True Label'], categories=labels_3, ordered=True)
    sns.violinplot(data=vplot_df, x='True Label', y='Polarity', palette=pal,
                   ax=axes[i], inner='quartile', cut=0)
    axes[i].axhline(0.6667, color='green', linestyle='--', linewidth=0.8, alpha=0.7)
    axes[i].axhline(0.3333, color='red',   linestyle='--', linewidth=0.8, alpha=0.7)
    axes[i].set_title(name)
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Polarity score [0–1]' if i % 4 == 0 else '')
axes[-1].set_visible(False)
fig.suptitle(
    'Polarity Score Distribution by True Participant Class\n'
    '(dashed lines = discretization thresholds: 0.333 / 0.667)',
    fontsize=12, fontweight='bold')
plt.tight_layout()
fig.savefig(os.path.join(FIGDIR, 'polarity_violin_by_class.pdf'), bbox_inches='tight')
fig.savefig(os.path.join(FIGDIR, 'polarity_violin_by_class.png'), dpi=300, bbox_inches='tight')
plt.show()

### Figure 10: Inter-Model Agreement (Pairwise Cohen's Kappa)

In [ ]:
n = len(display_names)
kappa_matrix = np.ones((n, n))
for i, col_i in enumerate(model_sentiment_cols):
    for j, col_j in enumerate(model_sentiment_cols):
        if i != j:
            kappa_matrix[i, j] = round(cohen_kappa_score(
                discretized_df[col_i].values,
                discretized_df[col_j].values), 3)

kappa_df = pd.DataFrame(kappa_matrix, index=display_names, columns=display_names)
fig, ax = plt.subplots(figsize=(8, 7))
mask = np.eye(n, dtype=bool)
sns.heatmap(kappa_df, annot=True, fmt='.3f', cmap='YlOrRd', ax=ax,
            vmin=0, vmax=1, square=True, mask=mask,
            linewidths=0.5, linecolor='white')
# fill diagonal manually
for k in range(n):
    ax.text(k+0.5, k+0.5, '1.000', ha='center', va='center', fontsize=9, color='white',
            fontweight='bold')
ax.set_title("Inter-Model Agreement — Pairwise Cohen's Kappa (3-class)", fontweight='bold')
plt.tight_layout()
fig.savefig(os.path.join(FIGDIR, 'intermodel_agreement.pdf'), bbox_inches='tight')
fig.savefig(os.path.join(FIGDIR, 'intermodel_agreement.png'), dpi=300, bbox_inches='tight')
plt.show()
print('\nKappa matrix:')
display(kappa_df)

### Figure 11: Text Features — Correct vs Misclassified

In [ ]:
err_plot = long_df.copy()
err_plot['Model'] = err_plot['model'].str.replace('clf_','').str.replace('_sentiment','')
err_plot['Classification'] = err_plot['sentiment'].map({1:'Correct', 0:'Misclassified'})

feat_cols  = ['word_count', 'char_count', 'adj_count', 'verb_count', 'noun_count']
feat_names = ['Word Count', 'Char Count', 'Adjective Count', 'Verb Count', 'Noun Count']

fig, axes = plt.subplots(1, 5, figsize=(16, 5))
for ax, feat, feat_name in zip(axes, feat_cols, feat_names):
    sns.boxplot(data=err_plot, x='Classification', y=feat, ax=ax,
                palette={'Correct':'steelblue','Misclassified':'tomato'},
                order=['Correct','Misclassified'], width=0.5,
                flierprops=dict(marker='o', markersize=2, alpha=0.4))
    ax.set_title(feat_name, fontweight='bold')
    ax.set_xlabel('')
    if ax != axes[0]:
        ax.set_ylabel('')
fig.suptitle('Text Features: Correctly Classified vs Misclassified (3-Class, all models)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
fig.savefig(os.path.join(FIGDIR, 'error_analysis_features.pdf'), bbox_inches='tight')
fig.savefig(os.path.join(FIGDIR, 'error_analysis_features.png'), dpi=300, bbox_inches='tight')
plt.show()

### Figure 12: Threshold Sensitivity Analysis

In [ ]:
pos_thresholds = np.linspace(0.50, 0.80, 61)
colors_t = plt.cm.tab10(np.linspace(0, 0.7, len(display_names)))

fig, ax = plt.subplots(figsize=(9, 5))
for name, pol_col, color in zip(display_names, polarity_cols, colors_t):
    pol = df[pol_col].values
    accs = []
    for pos_t in pos_thresholds:
        neg_t = 1 - pos_t
        pred = np.where(pol >= pos_t, 'positive',
               np.where(pol <= neg_t, 'negative', 'neutral'))
        accs.append(accuracy_score(y_true_3, pred))
    ax.plot(pos_thresholds, accs, label=name, color=color)

ax.axvline(0.6667, color='black', linestyle='--', linewidth=1.2,
           label='Current threshold (0.667)')
ax.set_xlabel('Positive threshold  (negative threshold = 1 − positive threshold)')
ax.set_ylabel('3-Class Accuracy')
ax.set_title('Threshold Sensitivity Analysis', fontweight='bold')
ax.legend(fontsize=7, ncol=2, loc='lower right')
plt.tight_layout()
fig.savefig(os.path.join(FIGDIR, 'threshold_sensitivity.pdf'), bbox_inches='tight')
fig.savefig(os.path.join(FIGDIR, 'threshold_sensitivity.png'), dpi=300, bbox_inches='tight')
plt.show()

## Additional Analysis
### Descriptive Statistics of Polarity Scores

In [ ]:
score_cols_all = ['participant_polarity','spaCy_polarity','nltk_polarity',
                  'nlptown_polarity','cardiff_polarity','tabmulti_polarity',
                  'hartmann_polarity','bhadresh_polarity']
score_labels = ['Participant','spaCy','NLTK','nlptown','Cardiff',
                'tabmulti','Hartmann','Bhadresh']

desc = df[score_cols_all].describe().T
desc.index = score_labels
desc['skewness'] = df[score_cols_all].skew().values
desc['median']   = df[score_cols_all].median().values
desc = desc[['count','mean','std','median','min','max','skewness']].round(4)
desc.to_csv(os.path.join(OUTDIR, 'descriptive_stats.csv'))
print('Saved descriptive_stats.csv')
display(desc)

### Continuous Score Correlation Matrix (Spearman)

In [ ]:
from scipy.stats import spearmanr as _sp

score_data = df[score_cols_all].values
n_scores   = len(score_cols_all)
corr_mat   = np.zeros((n_scores, n_scores))
pval_mat   = np.zeros((n_scores, n_scores))

for i in range(n_scores):
    for j in range(n_scores):
        r, p = _sp(score_data[:, i], score_data[:, j])
        corr_mat[i, j] = round(r, 3)
        pval_mat[i, j] = p

corr_df = pd.DataFrame(corr_mat, index=score_labels, columns=score_labels)
corr_df.to_csv(os.path.join(OUTDIR, 'correlation_matrix.csv'))

fig, ax = plt.subplots(figsize=(9, 7))
mask_upper = np.triu(np.ones_like(corr_mat, dtype=bool), k=1)
sns.heatmap(corr_df, annot=True, fmt='.3f', cmap='coolwarm', ax=ax,
            vmin=-1, vmax=1, square=True, linewidths=0.4,
            mask=mask_upper)

# add significance stars on lower triangle
for i in range(n_scores):
    for j in range(i):
        star = '***' if pval_mat[i,j] < 0.001 else ('**' if pval_mat[i,j] < 0.01
               else ('*' if pval_mat[i,j] < 0.05 else ''))
        if star:
            ax.text(j+0.75, i+0.75, star, ha='center', va='center',
                    fontsize=6, color='black')

ax.set_title('Spearman Correlation — All Continuous Polarity Scores\n'
             '(* p<0.05  ** p<0.01  *** p<0.001)',
             fontweight='bold')
plt.tight_layout()
fig.savefig(os.path.join(FIGDIR, 'correlation_matrix.pdf'), bbox_inches='tight')
fig.savefig(os.path.join(FIGDIR, 'correlation_matrix.png'), dpi=300, bbox_inches='tight')
plt.show()
print('Saved: correlation_matrix')

### McNemar's Pairwise Significance Test (Model vs Model)

In [ ]:
def mcnemar_p(pred_i, pred_j, y_true):
    ci = (pred_i == y_true)
    cj = (pred_j == y_true)
    b  = (ci & ~cj).sum()
    c  = (~ci & cj).sum()
    if b + c == 0:
        return 1.0
    chi2_stat = (abs(b - c) - 1)**2 / (b + c)  # continuity-corrected
    return float(_chi2.sf(chi2_stat, df=1))

n_m = len(display_names)
mcn_pval_raw = np.ones((n_m, n_m))
for i, col_i in enumerate(model_sentiment_cols):
    for j, col_j in enumerate(model_sentiment_cols):
        if i != j:
            mcn_pval_raw[i, j] = mcnemar_p(discretized_df[col_i].values,
                                            discretized_df[col_j].values,
                                            y_true_3)

# --- Holm-Bonferroni adjustment over the 21 unique pairs (upper triangle).
# Holm is uniformly more powerful than plain Bonferroni while keeping FWER ≤ 0.05.
pairs_idx = [(i, j) for i in range(n_m) for j in range(i+1, n_m)]
raw_ps    = [mcn_pval_raw[i, j] for (i, j) in pairs_idx]
order     = np.argsort(raw_ps)
n_pairs   = len(pairs_idx)
holm_ps   = np.ones(n_pairs)
running_max = 0.0
for rank, idx in enumerate(order):
    adj = (n_pairs - rank) * raw_ps[idx]
    running_max = max(running_max, adj)
    holm_ps[idx] = min(1.0, running_max)

# Symmetric matrix of Holm-adjusted p-values
mcn_pval = np.ones((n_m, n_m))
for k, (i, j) in enumerate(pairs_idx):
    mcn_pval[i, j] = round(holm_ps[k], 4)
    mcn_pval[j, i] = round(holm_ps[k], 4)

n_sig_holm = int(sum(p < 0.05 for p in holm_ps))
print(f'McNemar pairs: {n_pairs}  |  Holm-adjusted significant @0.05: {n_sig_holm}/{n_pairs}')

mcn_df = pd.DataFrame(mcn_pval, index=display_names, columns=display_names)
mcn_df.to_csv(os.path.join(OUTDIR, 'mcnemar_pairwise.csv'))

mcn_df_raw = pd.DataFrame(np.round(mcn_pval_raw, 4), index=display_names, columns=display_names)
mcn_df_raw.to_csv(os.path.join(OUTDIR, 'mcnemar_pairwise_raw.csv'))

log_p = -np.log10(np.where(mcn_pval == 0, 1e-10, mcn_pval))
np.fill_diagonal(log_p, 0)
mask_diag = np.eye(n_m, dtype=bool)

annot = np.empty((n_m, n_m), dtype=object)
for i in range(n_m):
    for j in range(n_m):
        if i == j:
            annot[i, j] = '—'
        else:
            p = mcn_pval[i, j]
            star = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
            annot[i, j] = f'{p:.3f}\n{star}'

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(log_p, annot=annot, fmt='', cmap='YlOrRd', ax=ax,
            xticklabels=display_names, yticklabels=display_names,
            square=True, linewidths=0.4, mask=mask_diag,
            cbar_kws={'label': '-log10(Holm-adj p)'})
ax.set_title(
    "McNemar's Pairwise Test — 3-Class (row vs col)\n"
    "Holm-Bonferroni adjusted; ns p≥0.05  * p<0.05  ** p<0.01  *** p<0.001",
    fontweight='bold')
plt.tight_layout()
fig.savefig(os.path.join(FIGDIR, 'mcnemar_pairwise.pdf'), bbox_inches='tight')
fig.savefig(os.path.join(FIGDIR, 'mcnemar_pairwise.png'), dpi=300, bbox_inches='tight')
plt.show()
print('Saved: mcnemar_pairwise (Holm-adjusted) and mcnemar_pairwise_raw (raw)')


### Qualitative Examples
Representative cases: unanimous correct, unanimous wrong, and split predictions per class.
Saved to `qualitative_examples.csv` — select rows for the paper table manually.

In [ ]:
clf_cols = [f'clf_{m}' for m in model_sentiment_cols]
pred_cols = model_sentiment_cols

sub_ex = sub_df.copy()
sub_ex['n_correct'] = (sub_ex[clf_cols] == 'correct').sum(axis=1)

examples = []
for true_cls in ['positive', 'neutral', 'negative']:
    subset = sub_ex[sub_ex['participant_sentiment'] == true_cls].copy()
    subset = subset.sort_values('word_count')  # prefer shorter texts

    # unanimous correct
    pool = subset[subset['n_correct'] == len(model_sentiment_cols)]
    if len(pool):
        row = pool.iloc[0].to_dict()
        row['example_type'] = f'{true_cls} | all correct'
        examples.append(row)

    # unanimous wrong
    pool = subset[subset['n_correct'] == 0]
    if len(pool):
        row = pool.iloc[0].to_dict()
        row['example_type'] = f'{true_cls} | all wrong'
        examples.append(row)

    # split — half correct, half wrong (±1)
    mid = len(model_sentiment_cols) // 2
    pool = subset[subset['n_correct'].between(mid - 1, mid + 1)]
    if len(pool):
        row = pool.iloc[0].to_dict()
        row['example_type'] = f'{true_cls} | models split'
        examples.append(row)

qual_df = pd.DataFrame(examples)

# Keep only readable columns
keep = ['example_type', 'corpus', 'participant_sentiment',
        'n_correct'] + pred_cols
qual_df = qual_df[keep].copy()
qual_df.columns = (['Example Type', 'Memory Text', 'Participant Label', '# Correct']
                   + display_names)

# Truncate long texts for display
qual_df['Memory Text'] = qual_df['Memory Text'].str[:200]

qual_df.to_csv(os.path.join(OUTDIR, 'qualitative_examples.csv'), index=False)
print(f'Saved qualitative_examples.csv  ({len(qual_df)} examples)')
print('Review the CSV and pick the best rows for your paper table.\n')
display(qual_df[['Example Type','Memory Text','Participant Label','# Correct'] + display_names])

### Bootstrap Confidence Intervals (95%, n=1000 resamples)
Provides uncertainty bounds for all key metrics. Uses percentile bootstrap — no normality assumption required.

In [ ]:
# --- Cluster (block) bootstrap: resample PARTICIPANTS, not memories.
# Each participant contributed multiple memories, so memory-level i.i.d.
# resampling underestimates standard errors. We resample participants with
# replacement and keep all of each participant's memories. We also use the
# SAME bootstrap samples across all 7 models so CIs are comparable across
# models (paired bootstrap), enabling honest CI-overlap reasoning.

participants = df['Participant'].values
unique_participants = np.unique(participants)
participant_to_indices = {p: np.where(participants == p)[0] for p in unique_participants}

rng = np.random.RandomState(42)
N_BOOT = 1000

boot_indices = []
for _ in range(N_BOOT):
    sampled_p = rng.choice(unique_participants, len(unique_participants), replace=True)
    idx = np.concatenate([participant_to_indices[p] for p in sampled_p])
    boot_indices.append(idx)

# Sanity check
print(f'Cluster bootstrap: {len(unique_participants)} participants, '
      f'{N_BOOT} resamples; mean resample size = '
      f'{np.mean([len(i) for i in boot_indices]):.0f} memories.')

ci_rows_3, ci_rows_bin = [], []

for name, col, pol_col in zip(display_names, model_sentiment_cols, polarity_cols):
    y_pred_3   = discretized_df[col].values
    y_pred_bin = np.array([bin_conv(y) for y in y_pred_3])
    y_pred_b01 = (y_pred_bin == 'positive').astype(int)
    pol        = df[pol_col].values

    acc3, f1m3, f1w3, kap3, bal3, wkap3 = [], [], [], [], [], []
    accb, f1mb, f1pb, aucb, kapb, balb = [], [], [], [], [], []

    for idx in boot_indices:                       # SAME indices for every model
        yt3   = y_true_3[idx];    yp3  = y_pred_3[idx]
        ytb   = y_true_bin[idx];  ypb  = y_pred_bin[idx]
        yt01  = y_true_b01[idx];  yp01 = y_pred_b01[idx]
        pol_i = pol[idx]

        acc3.append(accuracy_score(yt3, yp3))
        f1m3.append(f1_score(yt3, yp3, average='macro',    zero_division=0))
        f1w3.append(f1_score(yt3, yp3, average='weighted', zero_division=0))
        kap3.append(cohen_kappa_score(yt3, yp3))
        wkap3.append(cohen_kappa_score(yt3, yp3, weights='linear'))
        bal3.append(balanced_accuracy_score(yt3, yp3))

        accb.append(accuracy_score(ytb, ypb))
        f1mb.append(f1_score(ytb, ypb, average='macro',    zero_division=0))
        f1pb.append(f1_score(yt01, yp01, zero_division=0))
        try:
            aucb.append(roc_auc_score(yt01, pol_i))
        except Exception:
            aucb.append(np.nan)
        kapb.append(cohen_kappa_score(ytb, ypb))
        balb.append(balanced_accuracy_score(ytb, ypb))

    def ci(v):
        lo, hi = np.nanpercentile(v, 2.5), np.nanpercentile(v, 97.5)
        return round(lo, 4), round(hi, 4)

    ci_rows_3.append({'Model': name,
        'Accuracy_lo':       ci(acc3)[0],  'Accuracy_hi':       ci(acc3)[1],
        'Macro F1_lo':       ci(f1m3)[0],  'Macro F1_hi':       ci(f1m3)[1],
        'Weighted F1_lo':    ci(f1w3)[0],  'Weighted F1_hi':    ci(f1w3)[1],
        'Kappa_lo':          ci(kap3)[0],  'Kappa_hi':          ci(kap3)[1],
        'Weighted Kappa_lo': ci(wkap3)[0], 'Weighted Kappa_hi': ci(wkap3)[1],
        'Balanced Acc_lo':   ci(bal3)[0],  'Balanced Acc_hi':   ci(bal3)[1],
    })
    ci_rows_bin.append({'Model': name,
        'Accuracy_lo':       ci(accb)[0],   'Accuracy_hi':       ci(accb)[1],
        'Macro F1_lo':       ci(f1mb)[0],   'Macro F1_hi':       ci(f1mb)[1],
        'F1 positive_lo':    ci(f1pb)[0],   'F1 positive_hi':    ci(f1pb)[1],
        'AUC-ROC_lo':        ci(aucb)[0],   'AUC-ROC_hi':        ci(aucb)[1],
        'Kappa_lo':          ci(kapb)[0],   'Kappa_hi':          ci(kapb)[1],
        'Balanced Acc_lo':   ci(balb)[0],   'Balanced Acc_hi':   ci(balb)[1],
    })
    print(f'{name} done.')

ci_3_df   = pd.DataFrame(ci_rows_3)
ci_bin_df = pd.DataFrame(ci_rows_bin)

full_3_df   = metrics_3_df.merge(ci_3_df, on='Model')
full_bin_df = metrics_bin_df.merge(ci_bin_df, on='Model')
full_3_df.to_csv(os.path.join(OUTDIR, 'metrics_3class_with_ci.csv'), index=False)
full_bin_df.to_csv(os.path.join(OUTDIR, 'metrics_binary_with_ci.csv'), index=False)

def fmt_ci(df, ci_df, metric):
    out = []
    for _, row in df.iterrows():
        m   = row[metric]
        lo  = ci_df.loc[ci_df['Model']==row['Model'], f'{metric}_lo'].values[0]
        hi  = ci_df.loc[ci_df['Model']==row['Model'], f'{metric}_hi'].values[0]
        out.append(f'{m:.3f} [{lo:.3f}\u2013{hi:.3f}]')
    return out

summary_3 = pd.DataFrame({'Model': metrics_3_df['Model']})
for m in ['Accuracy','Macro F1','Weighted F1','Kappa','Weighted Kappa','Balanced Acc']:
    summary_3[m] = fmt_ci(metrics_3_df, ci_3_df, m)

summary_bin = pd.DataFrame({'Model': metrics_bin_df['Model']})
for m in ['Accuracy','Macro F1','F1 positive','AUC-ROC','Kappa','Balanced Acc']:
    summary_bin[m] = fmt_ci(metrics_bin_df, ci_bin_df, m)

print('\n3-class metrics with 95% CI (cluster bootstrap, paired):')
display(summary_3)
print('\nBinary metrics with 95% CI (cluster bootstrap, paired):')
display(summary_bin)


### Figure 13: Accuracy & Macro F1 with 95% CI

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, mdf, ci_df, title in zip(
        axes,
        [metrics_3_df, metrics_bin_df],
        [ci_3_df,      ci_bin_df],
        ['3-Class', 'Binary']):

    y_pos = np.arange(len(display_names))
    for offset, metric, color in [(-0.18,'Accuracy','steelblue'),
                                    ( 0.18,'Macro F1', 'tomato')]:
        vals  = mdf[metric].values
        lo    = ci_df[f'{metric}_lo'].values
        hi    = ci_df[f'{metric}_hi'].values
        yerr  = np.array([vals - lo, hi - vals])
        ax.barh(y_pos + offset, vals, height=0.3, xerr=yerr,
                color=color, alpha=0.8, capsize=4,
                error_kw={'linewidth': 1.2}, label=metric)

    ax.set_yticks(y_pos)
    ax.set_yticklabels(display_names)
    ax.set_xlabel('Score')
    ax.set_xlim(0, 1.05)
    ax.set_title(f'{title} — Accuracy & Macro F1 (95% CI)', fontweight='bold')
    ax.legend(loc='lower right')
    ax.axvline(0, color='black', linewidth=0.6)

plt.tight_layout()
fig.savefig(os.path.join(FIGDIR, 'metrics_with_ci.pdf'), bbox_inches='tight')
fig.savefig(os.path.join(FIGDIR, 'metrics_with_ci.png'), dpi=300, bbox_inches='tight')
plt.show()
print('Saved: metrics_with_ci')

### McNemar's Effect Size — Phi Coefficient (φ)
Phi measures the **strength** of the difference between model pairs, independent of sample size.  
|φ| < 0.1 = negligible, 0.1–0.3 = small, 0.3–0.5 = moderate, > 0.5 = large.

In [ ]:
phi_matrix = np.zeros((n_m, n_m))

for i, col_i in enumerate(model_sentiment_cols):
    for j, col_j in enumerate(model_sentiment_cols):
        if i == j:
            phi_matrix[i, j] = 1.0
            continue
        ci_ = (discretized_df[col_i].values == y_true_3)
        cj_ = (discretized_df[col_j].values == y_true_3)
        a = ( ci_ &  cj_).sum()
        b = ( ci_ & ~cj_).sum()
        c = (~ci_ &  cj_).sum()
        d = (~ci_ & ~cj_).sum()
        denom = np.sqrt((a+b)*(c+d)*(a+c)*(b+d))
        phi_matrix[i, j] = round((a*d - b*c) / denom, 3) if denom > 0 else 0.0

phi_df = pd.DataFrame(phi_matrix, index=display_names, columns=display_names)
phi_df.to_csv(os.path.join(OUTDIR, 'mcnemar_effect_size_phi.csv'))

fig, ax = plt.subplots(figsize=(8, 7))
mask_diag = np.eye(n_m, dtype=bool)
sns.heatmap(phi_df, annot=True, fmt='.3f', cmap='RdYlGn', ax=ax,
            vmin=-1, vmax=1, square=True, linewidths=0.4,
            mask=mask_diag)
for k in range(n_m):
    ax.text(k+0.5, k+0.5, '1.000', ha='center', va='center',
            fontsize=9, fontweight='bold', color='gray')
ax.set_title("McNemar's Effect Size — Phi Coefficient (φ)\n"
             '|φ|<0.1 negligible · 0.1–0.3 small · 0.3–0.5 moderate · >0.5 large',
             fontweight='bold')
plt.tight_layout()
fig.savefig(os.path.join(FIGDIR, 'mcnemar_effect_phi.pdf'), bbox_inches='tight')
fig.savefig(os.path.join(FIGDIR, 'mcnemar_effect_phi.png'), dpi=300, bbox_inches='tight')
plt.show()
print('Saved: mcnemar_effect_phi  |  mcnemar_effect_size_phi.csv')

## Revision-round additions

The following analyses were added in response to a methodological review:

1. **Ensemble baseline** — mean of the seven normalised model polarity scores, then
   discretised the same way as the participant ground truth.
2. **TF-IDF + Logistic Regression baseline** — supervised upper bound estimated
   with leave-one-participant-out cross-validation so no participant's text leaks
   between train and test.
3. **Per-participant accuracy distribution** — shows whether the headline
   accuracy reflects most participants or is dragged by a few.
4. **Length-stratified metrics** — accuracy by memory word-count quartile.
5. **Valence-weight sensitivity** — re-evaluates Hartmann and Bhadresh with
   randomly perturbed NRC-VAD weights to confirm conclusions are not driven by
   the specific valence values chosen.


In [ ]:
# --- Ensemble baseline: mean of the 7 model polarity scores in [0,1].
ensemble_polarity = df[polarity_cols].mean(axis=1).values
df['ensemble_polarity'] = ensemble_polarity

ensemble_pred_3 = np.array([disc_conv(x) for x in ensemble_polarity])
ensemble_pred_bin = np.array([bin_conv(x) for x in ensemble_pred_3])
ensemble_pred_b01 = (ensemble_pred_bin == 'positive').astype(int)

ens_row_3 = {
    'Model':          'Ensemble (mean)',
    'Accuracy':       round(accuracy_score(y_true_3, ensemble_pred_3), 4),
    'Macro F1':       round(f1_score(y_true_3, ensemble_pred_3, average='macro', zero_division=0), 4),
    'Weighted F1':    round(f1_score(y_true_3, ensemble_pred_3, average='weighted', zero_division=0), 4),
    'Kappa':          round(cohen_kappa_score(y_true_3, ensemble_pred_3), 4),
    'Weighted Kappa': round(cohen_kappa_score(y_true_3, ensemble_pred_3, weights='linear'), 4),
    'MCC':            round(matthews_corrcoef(y_true_3, ensemble_pred_3), 4),
    'Spearman r':     round(spearmanr(participant_cont, ensemble_polarity)[0], 4),
    'Balanced Acc':   round(balanced_accuracy_score(y_true_3, ensemble_pred_3), 4),
}
ens_row_bin = {
    'Model':          'Ensemble (mean)',
    'Accuracy':       round(accuracy_score(y_true_bin, ensemble_pred_bin), 4),
    'Macro F1':       round(f1_score(y_true_bin, ensemble_pred_bin, average='macro', zero_division=0), 4),
    'F1 positive':    round(f1_score(y_true_b01, ensemble_pred_b01, zero_division=0), 4),
    'AUC-ROC':        round(roc_auc_score(y_true_b01, ensemble_polarity), 4),
    'Kappa':          round(cohen_kappa_score(y_true_bin, ensemble_pred_bin), 4),
    'Balanced Acc':   round(balanced_accuracy_score(y_true_bin, ensemble_pred_bin), 4),
}

ensemble_df = pd.DataFrame([ens_row_3])
ensemble_bin_df = pd.DataFrame([ens_row_bin])
ensemble_df.to_csv(os.path.join(OUTDIR, 'ensemble_baseline_3class.csv'), index=False)
ensemble_bin_df.to_csv(os.path.join(OUTDIR, 'ensemble_baseline_binary.csv'), index=False)

print('Ensemble baseline (mean of 7 model polarity scores):')
print('  3-class :', ens_row_3)
print('  binary  :', ens_row_bin)

# Compare with best single model
best_3 = metrics_3_df.loc[metrics_3_df['Macro F1'].idxmax()]
best_b = metrics_bin_df.loc[metrics_bin_df['Macro F1'].idxmax()]
print(f"\nBest single (3-class)  : {best_3['Model']:>15s}  macro F1 = {best_3['Macro F1']:.4f}")
print(f"Ensemble (3-class)     : {'mean':>15s}  macro F1 = {ens_row_3['Macro F1']:.4f}")
print(f"\nBest single (binary)   : {best_b['Model']:>15s}  macro F1 = {best_b['Macro F1']:.4f}")
print(f"Ensemble (binary)      : {'mean':>15s}  macro F1 = {ens_row_bin['Macro F1']:.4f}")


In [ ]:
# --- TF-IDF + Logistic Regression baseline, leave-one-PARTICIPANT-out CV.
# Every participant's memories appear ONLY in either train or test, never
# both, so this is a strict measure of out-of-participant generalisation.
# This is the supervised upper bound for what a small in-domain learner can
# achieve on this corpus.

corpus_texts = df['corpus'].astype(str).values
y3 = y_true_3
groups = df['Participant'].values
unique_p = np.unique(groups)
print(f'LeaveOneGroupOut CV: {len(unique_p)} participants, {len(corpus_texts)} memories')

logo = LeaveOneGroupOut()
all_preds_3 = np.empty(len(y3), dtype=object)
all_pred_pol = np.zeros(len(y3))   # P(positive) - P(negative) as continuous proxy

# Fit per fold. Use ngrams 1-2 with a modest vocabulary cap.
fold_count = 0
for train_idx, test_idx in logo.split(corpus_texts, y3, groups):
    fold_count += 1
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.95,
                          sublinear_tf=True, max_features=20000,
                          stop_words='english')
    Xtr = vec.fit_transform(corpus_texts[train_idx])
    Xte = vec.transform(corpus_texts[test_idx])
    clf = LogisticRegression(max_iter=2000, C=1.0, class_weight='balanced',
                             multi_class='auto', solver='liblinear')
    # liblinear doesn't support multinomial; switch to lbfgs for >2 classes
    clf = LogisticRegression(max_iter=2000, C=1.0, class_weight='balanced',
                             solver='lbfgs')
    clf.fit(Xtr, y3[train_idx])
    preds = clf.predict(Xte)
    all_preds_3[test_idx] = preds
    # continuous proxy: P(pos) - P(neg)
    proba = clf.predict_proba(Xte)
    classes = list(clf.classes_)
    if 'positive' in classes and 'negative' in classes:
        all_pred_pol[test_idx] = proba[:, classes.index('positive')] - proba[:, classes.index('negative')]
    if fold_count % 50 == 0:
        print(f'  fold {fold_count}/{len(unique_p)}')

all_preds_bin = np.array([bin_conv(y) for y in all_preds_3])
all_preds_b01 = (all_preds_bin == 'positive').astype(int)

# Per-class metrics
lr_3 = {
    'Model':         'TF-IDF+LR (LOPO)',
    'Accuracy':      round(accuracy_score(y_true_3, all_preds_3), 4),
    'Balanced Acc':  round(balanced_accuracy_score(y_true_3, all_preds_3), 4),
    'Macro F1':      round(f1_score(y_true_3, all_preds_3, average='macro', zero_division=0), 4),
    'Weighted F1':   round(f1_score(y_true_3, all_preds_3, average='weighted', zero_division=0), 4),
    'Kappa':         round(cohen_kappa_score(y_true_3, all_preds_3), 4),
    'Weighted Kappa':round(cohen_kappa_score(y_true_3, all_preds_3, weights='linear'), 4),
    'MCC':           round(matthews_corrcoef(y_true_3, all_preds_3), 4),
    'Spearman r':    round(spearmanr(participant_cont, all_pred_pol)[0], 4),
}
lr_bin = {
    'Model':       'TF-IDF+LR (LOPO)',
    'Accuracy':    round(accuracy_score(y_true_bin, all_preds_bin), 4),
    'Balanced Acc':round(balanced_accuracy_score(y_true_bin, all_preds_bin), 4),
    'Macro F1':    round(f1_score(y_true_bin, all_preds_bin, average='macro', zero_division=0), 4),
    'F1 positive': round(f1_score(y_true_b01, all_preds_b01, zero_division=0), 4),
    'Kappa':       round(cohen_kappa_score(y_true_bin, all_preds_bin), 4),
}
# AUC-ROC needs continuous score
try:
    lr_bin['AUC-ROC'] = round(roc_auc_score(y_true_b01, all_pred_pol), 4)
except Exception:
    lr_bin['AUC-ROC'] = float('nan')

pd.DataFrame([lr_3]).to_csv(os.path.join(OUTDIR, 'lr_baseline_3class.csv'), index=False)
pd.DataFrame([lr_bin]).to_csv(os.path.join(OUTDIR, 'lr_baseline_binary.csv'), index=False)

print('\nTF-IDF + Logistic Regression (LOPO):')
print('  3-class:', lr_3)
print('  binary :', lr_bin)


In [ ]:
# --- Per-participant accuracy distribution (best model = Cardiff).
# A single overall accuracy can mask large between-participant variation.
best_model_col = 'cardiff_sentiment'
per_p_records = []
for p in unique_participants:
    idx = np.where(participants == p)[0]
    yt  = y_true_3[idx]
    yp  = discretized_df[best_model_col].values[idx]
    if len(idx) >= 2:                # exclude participants with <2 memories
        per_p_records.append({
            'Participant': p,
            'n_memories':  len(idx),
            'accuracy':    accuracy_score(yt, yp),
        })
per_p_df = pd.DataFrame(per_p_records)
per_p_df.to_csv(os.path.join(OUTDIR, 'per_participant_accuracy_cardiff.csv'), index=False)

acc_vals = per_p_df['accuracy'].values
print(f'Per-participant accuracy (Cardiff, n={len(per_p_df)} participants):')
print(f'  median = {np.median(acc_vals):.3f}')
print(f'  mean   = {np.mean(acc_vals):.3f}  (overall pooled = {accuracy_score(y_true_3, discretized_df[best_model_col].values):.3f})')
print(f'  IQR    = [{np.percentile(acc_vals,25):.3f}, {np.percentile(acc_vals,75):.3f}]')
print(f'  range  = [{acc_vals.min():.3f}, {acc_vals.max():.3f}]')
print(f'  % participants with acc < 0.40: {(acc_vals < 0.40).mean()*100:.1f}%')
print(f'  % participants with acc > 0.80: {(acc_vals > 0.80).mean()*100:.1f}%')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(acc_vals, bins=20, color='steelblue', edgecolor='black', alpha=0.85)
ax.axvline(np.mean(acc_vals), color='red', linestyle='--', label=f'mean = {np.mean(acc_vals):.2f}')
ax.axvline(np.median(acc_vals), color='black', linestyle=':',  label=f'median = {np.median(acc_vals):.2f}')
ax.set_xlabel('Cardiff RoBERTa accuracy on each participant\'s memories')
ax.set_ylabel('Number of participants')
ax.set_title('Per-Participant Accuracy Distribution (Cardiff RoBERTa)', fontweight='bold')
ax.legend()
plt.tight_layout()
fig.savefig(os.path.join(FIGDIR, 'per_participant_accuracy.pdf'), bbox_inches='tight')
fig.savefig(os.path.join(FIGDIR, 'per_participant_accuracy.png'), dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# --- Length-stratified accuracy: does performance depend on memory length?
wc = df['word_count'].values
quartiles = np.percentile(wc, [25, 50, 75])
def bin_q(w):
    if w <= quartiles[0]: return 'Q1 (shortest)'
    if w <= quartiles[1]: return 'Q2'
    if w <= quartiles[2]: return 'Q3'
    return 'Q4 (longest)'
length_bin = np.array([bin_q(w) for w in wc])

length_records = []
for name, col in zip(display_names, model_sentiment_cols):
    yp = discretized_df[col].values
    for q in ['Q1 (shortest)', 'Q2', 'Q3', 'Q4 (longest)']:
        mask = length_bin == q
        length_records.append({
            'Model':    name,
            'Quartile': q,
            'n':        int(mask.sum()),
            'mean_words': round(wc[mask].mean(), 1),
            'accuracy': round(accuracy_score(y_true_3[mask], yp[mask]), 4),
            'macro_f1': round(f1_score(y_true_3[mask], yp[mask], average='macro', zero_division=0), 4),
        })
length_df = pd.DataFrame(length_records)
length_df.to_csv(os.path.join(OUTDIR, 'length_stratified_metrics.csv'), index=False)
print('Length-stratified metrics:')
print(length_df.to_string(index=False))

# Pivot to wide for plotting
pivot_acc = length_df.pivot(index='Model', columns='Quartile', values='accuracy')
pivot_acc = pivot_acc[['Q1 (shortest)', 'Q2', 'Q3', 'Q4 (longest)']]
pivot_acc = pivot_acc.reindex(display_names)

fig, ax = plt.subplots(figsize=(10, 5))
pivot_acc.plot(kind='bar', ax=ax, colormap='viridis', width=0.8, edgecolor='black')
ax.set_ylabel('3-class accuracy')
ax.set_title('Accuracy by Memory Length (Word-Count Quartile)', fontweight='bold')
ax.set_xticklabels(pivot_acc.index, rotation=30, ha='right')
ax.legend(title='Word-count quartile', fontsize=8, loc='lower right')
ax.set_ylim(0, 0.7)
plt.tight_layout()
fig.savefig(os.path.join(FIGDIR, 'length_stratified_accuracy.pdf'), bbox_inches='tight')
fig.savefig(os.path.join(FIGDIR, 'length_stratified_accuracy.png'), dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# --- Sensitivity analysis: Hartmann & Bhadresh valence weights.
# We perturb each NRC-VAD-derived weight by uniform noise in ±0.10 (clipped
# to [0,1]), re-derive the polarity score, re-discretise, and recompute
# 3-class macro F1. Repeating 200 times gives a distribution that says how
# much our reported F1 depends on the exact valence values chosen.

N_SENS = 200
rng_sens = np.random.RandomState(123)

def perturb(weights, noise=0.10):
    return {k: float(np.clip(v + rng_sens.uniform(-noise, noise), 0.0, 1.0))
            for k, v in weights.items()}

def f1_for_hartmann(weights):
    pol = np.array([sum(d['score'] * weights[d['label']] for d in dl) for dl in HARTMANN_RAW])
    pred = np.array([disc_conv(p) for p in pol])
    return f1_score(y_true_3, pred, average='macro', zero_division=0)

def f1_for_bhadresh(weights, thr):
    BH_POS = {'joy', 'love', 'surprise'}
    preds = []
    for d in BHADRESH_RAW:
        if d[0]['score'] < thr:
            preds.append('neutral')
        elif d[0]['label'] in BH_POS:
            preds.append('positive')
        else:
            preds.append('negative')
    return f1_score(y_true_3, preds, average='macro', zero_division=0)

hartmann_sens = [f1_for_hartmann(perturb(HARTMANN_VALENCE)) for _ in range(N_SENS)]
bhadresh_sens = [f1_for_bhadresh(perturb(BHADRESH_VALENCE), NEUTRAL_THRESHOLD) for _ in range(N_SENS)]

base_h = f1_for_hartmann(HARTMANN_VALENCE)
base_b = f1_for_bhadresh(BHADRESH_VALENCE, NEUTRAL_THRESHOLD)

sens_df = pd.DataFrame({
    'Model':       ['Hartmann', 'Bhadresh'],
    'baseline_f1': [round(base_h, 4), round(base_b, 4)],
    'sens_mean':   [round(np.mean(hartmann_sens), 4), round(np.mean(bhadresh_sens), 4)],
    'sens_std':    [round(np.std(hartmann_sens), 4),  round(np.std(bhadresh_sens), 4)],
    'sens_min':    [round(np.min(hartmann_sens), 4),  round(np.min(bhadresh_sens), 4)],
    'sens_max':    [round(np.max(hartmann_sens), 4),  round(np.max(bhadresh_sens), 4)],
    'sens_2.5%':   [round(np.percentile(hartmann_sens, 2.5), 4), round(np.percentile(bhadresh_sens, 2.5), 4)],
    'sens_97.5%':  [round(np.percentile(hartmann_sens, 97.5), 4), round(np.percentile(bhadresh_sens, 97.5), 4)],
})
sens_df.to_csv(os.path.join(OUTDIR, 'valence_sensitivity.csv'), index=False)
print('Valence-weight sensitivity (200 random perturbations of NRC-VAD weights, ±0.10):')
print(sens_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, vals, name, base in zip(axes,
                                 [hartmann_sens, bhadresh_sens],
                                 ['Hartmann', 'Bhadresh'],
                                 [base_h, base_b]):
    ax.hist(vals, bins=25, color='steelblue', edgecolor='black', alpha=0.85)
    ax.axvline(base, color='red', linestyle='--', linewidth=1.5,
               label=f'NRC-VAD baseline = {base:.3f}')
    ax.set_xlabel('3-class macro F1')
    ax.set_ylabel('Count')
    ax.set_title(f'{name} — F1 under perturbed valence weights', fontweight='bold')
    ax.legend()
plt.tight_layout()
fig.savefig(os.path.join(FIGDIR, 'valence_sensitivity.pdf'), bbox_inches='tight')
fig.savefig(os.path.join(FIGDIR, 'valence_sensitivity.png'), dpi=300, bbox_inches='tight')
plt.show()


---
## Pipeline complete
All outputs are in `data/results/`. Figures are in `data/results/figures/`.

In [ ]:
import os
print('=== Output files ===')
for f in sorted(os.listdir(OUTDIR)):
    print(f'  {f}')
print(f'\n=== Figures ===')
for f in sorted(os.listdir(FIGDIR)):
    print(f'  {f}')
print(f'\nDone. All outputs saved to {OUTDIR}')